# Knowledge Graph Creation

In [1]:
import json
import sys
import pandas as pd
import collections 
import os
import numpy as np
from itertools import chain
from itertools import combinations
sys.path.insert(0, '..')
from src.experiment_utils.helper_classes import token, span, repository
from src.d02_corpus_statistics.corpus import Corpus
import types
from owlready2 import sync_reasoner

cwd = os.getcwd()
pol_dir = cwd+"/../src/d01_data"

In [2]:
pol_df = pd.read_pickle(pol_dir+"/preprocessed_dataframe.pkl")[["Policy","Text","Tokens","Curation"]]
meta_df = pd.read_csv(pol_dir+"/EU_metadata.csv", delimiter=";")

In [5]:
pol_df.loc["EU_32018R1999_Title_0_Chapter_7_Section_3_Article_43"]

Policy                                                       
Text        article 43\r\nexercise of the delegation\r\n1....
Tokens      [token id: T1, start:0 stop:7 text:article tag...
Curation    [span id:CUR0 annotator:Curation layer:Instrum...
Name: EU_32018R1999_Title_0_Chapter_7_Section_3_Article_43, dtype: object

In [6]:
ftr_tkn_cnt = {}
for artid in pol_df.index:
    for spn in pol_df.loc[artid, "Curation"]:
        try:
            ftr_tkn_cnt[spn.feature].append(len(spn.text.split(" ")))
        except:
            ftr_tkn_cnt[spn.feature] = [len(spn.text.split(" "))]

In [13]:
'''
for ftr in ["InstrumentType", "Actor", "Time", "Objective", "Resource"]:
    print("\n", ftr)
    print(np.min(ftr_tkn_cnt[ftr]))    
    print(np.mean(ftr_tkn_cnt[ftr]))
    print(np.max(ftr_tkn_cnt[ftr]))
'''
from collections import Counter
for ftr in ["InstrumentType", "Actor", "Time", "Objective", "Resource"]:
    print("\n", ftr)
    print(Counter(ftr_tkn_cnt[ftr]))

#import matplotlib.pyplot as plt

#plt.hist(ftr_tkn_cnt["InstrumentType"])


 InstrumentType
Counter({1: 1487, 2: 880, 3: 448, 4: 220, 5: 130, 6: 119, 7: 39, 8: 23, 9: 12, 11: 8, 10: 8, 12: 2, 16: 2, 15: 2, 14: 1, 52: 1, 18: 1, 22: 1, 23: 1, 25: 1, 13: 1, 17: 1})

 Actor
Counter({2: 2510, 1: 2371, 3: 668, 4: 155, 5: 134, 6: 58, 7: 36, 10: 25, 8: 19, 9: 17, 11: 7, 12: 4, 13: 3, 15: 3, 20: 3, 18: 3, 14: 2, 27: 1, 19: 1, 16: 1})

 Time
Counter({1: 230, 2: 197, 3: 140, 4: 52, 5: 36, 6: 30, 8: 22, 7: 20, 9: 19, 10: 10, 15: 8, 12: 7, 13: 5, 22: 4, 17: 3, 14: 3, 19: 3, 16: 3, 21: 2, 20: 1, 26: 1, 25: 1, 11: 1})

 Objective
Counter({3: 142, 2: 107, 1: 99, 4: 77, 5: 76, 6: 58, 8: 58, 7: 53, 9: 47, 11: 38, 10: 28, 12: 24, 13: 23, 14: 20, 15: 14, 16: 14, 22: 13, 19: 11, 20: 9, 23: 9, 25: 8, 17: 8, 24: 8, 21: 8, 30: 7, 31: 7, 18: 6, 32: 3, 28: 2, 27: 2, 26: 2, 47: 2, 36: 2, 37: 1, 39: 1, 49: 1, 46: 1, 34: 1, 35: 1, 43: 1})

 Resource
Counter({1: 238, 2: 141, 4: 42, 3: 38, 5: 17, 6: 14, 8: 6, 7: 4, 10: 3, 11: 3, 25: 1, 48: 1, 21: 1, 13: 1, 33: 1})


Each policy has chapters, each chapter has sections, each section has articles
Each article has spans
Each span has text labeled with a tag, which is a specified version of a feature which is a specified version of a layer

In [3]:
from owlready2 import *

owl_path = cwd+"/auxil/ontology.owl"
onto = get_ontology(owl_path).load()

with onto:
    # classes
    #policy structure
    class Policy(Thing): 
        pass
    class Chapter(Thing): 
        pass
    class Section(Thing): 
        pass
    class Article(Thing): 
        pass
    #spans
    class Layer(Thing): 
        pass
    class Feature(Thing): 
        pass
    class Tag(Thing): 
        pass
    class Span(Thing): 
        pass

    # object properties
    #structure
    # should I change these to "part of" and "type of" instead of specific relationships??
    class hasPart(ObjectProperty):
        domain = [Policy, Chapter, Section, Article]
        range = [Chapter, Section, Article, Span]
    class partOf(ObjectProperty):
        domain = [Chapter, Section, Article, Span]
        range = [Policy, Chapter, Section, Article]
    class hasChapter(hasPart):
        domain = [Policy]
        range = [Chapter]
    class hasSection(hasPart):
        domain = [Chapter]
        range = [Section]
    class hasArticle(hasPart):
        domain = [Section]
        range = [Article]
    class hasSpan(hasPart):
        domain = [Article]
        range = [Span]
    class isChapterOf(partOf):
        domain = [Chapter]
        range = [Policy]
        inverse_property = hasChapter
    class isSectionOf(partOf):
        domain = [Section]
        range = [Chapter]
        inverse_property = hasSection
    class isArticleOf(partOf):
        domain = [Article]
        range = [Section]
        inverse_property = hasArticle
    class isSpanOf(partOf):
        domain = [Span]
        range = [Article]
        inverse_property = hasSpan
    #spans
    class inTag(ObjectProperty):
        domain = [Span]
        range = [Tag]
    class taggedSpan(ObjectProperty):
        domain = [Tag]
        range = [Span]
        inverse_property = inTag
    ### ??????????????
    ### do i keep these here or specify strict elsewhere?
    ### since tags always have the same feature which always have the same layer
    class inLayer(ObjectProperty):
        domain = [Feature]
        range = [Layer]
    class inFeature(ObjectProperty):
        domain = [Tag]
        range = [Feature]
    class hasFeature(ObjectProperty):
        domain = [Layer]
        range = [Feature]
        inverse_property = inLayer
    class hasTag(ObjectProperty):
        domain = [Feature]
        range = [Tag]
        inverse_property = inFeature
    
    # data properties
    #policy structure
    class policy_code(DataProperty):
        domain = [Policy]
        range = [str]
    class chapter_num(DataProperty):
        domain = [Chapter]
        range = [str]
    class section_num(DataProperty):
        domain = [Section]
        range = [str]
    class article_num(DataProperty):
        domain = [Article]
        range = [str]
    class article_text(DataProperty):
        domain = [Article]
        range = [str]
    #spans
    class layer_name(DataProperty):
        domain = [Layer]
        range = [str]
    class feature_name(DataProperty):
        domain = [Feature]
        range = [str]
    class tag_name(DataProperty):
        domain = [Tag]
        range = [str]
    class span_id(DataProperty):
        domain = [Span]
        range = [str]
    class span_text(DataProperty):
        domain = [Span]
        range = [str]

## KG from dataset
Each policy has articles
Each article has features, instruments and list(policy_design_tags)
Each feature has a tag with a span (or a span with a tag)

In [4]:
for i in Thing.instances():
    destroy_entity(i)
    #print(i)

In [5]:
# to reexamine, currently excluding:
# articles with id's [policy_code]_Whereas or [policy_code]_front
works = []
fails = []
for i in range(len(list(pol_df.index))):
    #x.append(pol_df.index[i].split("_")[3])
    try:
        pol_df.index[i].split("_")[3]
        works.append(i)
    except:
        fails.append(i)
#pol_df.iloc[fails]

In [6]:
#policies = {}
#chapters = {}
#sections = {}
#articles = {}

for ind in pol_df.index:
    deets = ind.split("_")
    policy_code = "_".join(deets[:2])
    #ignore whereas and front bits for now
    if deets[2] == "Whereas" or deets[2] == "front":
        continue
    chapter_num = deets[5]
    section_num = deets[7]
    article_num = deets[9]
    # Policy
    policy = onto[policy_code]
    if not policy:
        policy = onto.Policy(policy_code) #unique Policy instance with name of policy_code
        policy.policy_code = [policy_code] #sets the Policy instance's policy_code to the policy_code
        #policies[policy_code] = policy #stores the Policy instance in the dictionary
    # Chapter
    chapter_key = f"{policy_code}_Chapter_{chapter_num}" #makes unique chapter key
    chapter = onto[chapter_key]
    if not chapter:
        chapter = onto.Chapter(chapter_key)
        chapter.chapter_num = [chapter_num]
        chapter.isChapterOf.append(policy)
        #chapters[chapter_key] = chapter
    # Section
    section_key = f"{policy_code}_Chapter_{chapter_num}_Section_{section_num}"
    section = onto[section_key]
    if not section:
        section = onto.Section(section_key)
        section.section_num = [section_num]
        section.isSectionOf.append(chapter)
        #sections[section_key] = section
    # Article
    article_key = f"{policy_code}_Chapter_{chapter_num}_Section_{section_num}_Article_{article_num}"
    article = onto[article_key]
    if not article:
        article = onto.Article(article_key)
        article.article_num = [article_num]
        article.isArticleOf.append(section)
        article.article_text = [pol_df.loc[ind, "Text"]]
        #articles[article_key] = article

In [7]:
for ind in pol_df.index:
    if "EU_32018R1999" in ind:
        if "_Chapter_6_Section_3_Article_40" in ind:
            print(pol_df.loc[ind])

Policy                                                       
Text        article 40\r\nestablishment and operation of r...
Tokens      [token id: T214199, start:0 stop:7 text:articl...
Curation    [span id:CUR17975 annotator:Curation layer:Ins...
Name: EU_32018R1999_Title_0_Chapter_6_Section_3_Article_40, dtype: object


In [8]:
no_tag_lst = []
for ind in pol_df.index:
    deets = ind.split("_")
    policy_code = "_".join(deets[:2])
    #ignore whereas and front bits for now
    if deets[2] == "Whereas" or deets[2] == "front":
        continue
    chapter_num = deets[5]
    section_num = deets[7]
    article_num = deets[9]
    article_key = f"{policy_code}_Chapter_{chapter_num}_Section_{section_num}_Article_{article_num}"

    article = onto[article_key]
    for spanobj in pol_df.loc[ind, "Curation"]:
        span_id = f"{article_key}_{spanobj.span_id}"
        if not spanobj.tag:
            no_tag_lst.append((ind, spanobj.span_id))
            continue
        if spanobj.feature == "Technologyandapplicationspecificity":
            continue
        span = onto.Span(span_id+"_span")
        span.span_id = [span_id]
        span.span_text = [spanobj.text]
        # tag
        tag = onto[spanobj.tag]
        if not tag:
            tag = onto.Tag(spanobj.tag)
            tag.tag_name = [spanobj.tag]
        # can i assert feature/layer inheritance globally? should i?
        feature = onto[spanobj.feature]
        if not feature:
            feature = onto.Feature(spanobj.feature)
            feature.feature_name = [spanobj.feature]
        layer = onto[spanobj.layer]
        if not layer:
            layer = onto.Layer(spanobj.layer)
            layer.layer_name = [spanobj.layer]
        span.inTag = [tag]
        tag.inFeature = [feature]
        feature.inLayer = [layer]
        #tag.taggedSpan.append(span)
        span.isSpanOf = [article]
        #article.hasSpan.append(span)
    #articles[article_key] = article

#onto.save(file=f"{cwd}/auxil/policy_kg_populated.owl", format="rdfxml")

In [9]:
sync_reasoner(infer_property_values=True)
onto.save(file=f"{cwd}/auxil/policy_kg_populated.owl", format="rdfxml")

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp C:\Users\allie\AppData\Roaming\Python\Python311\site-packages\owlready2\hermit;C:\Users\allie\AppData\Roaming\Python\Python311\site-packages\owlready2\hermit\HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:///C:/Users/allie/AppData/Local/Temp/tmpv8akmoxg -Y
* Owlready2 * HermiT took 6.181339740753174 seconds
* Owlready * Reparenting ontology.EU_32012L0027_Chapter_1_Section_0_Article_02_CUR4009_span: {ontology.Span} => {ontology.Article, ontology.Span, ontology.Section, ontology.Chapter}
* Owlready * Reparenting ontology.EU_32014R0421_Chapter_0_Section_0_Article_01_CUR20222_span: {ontology.Span} => {ontology.Article, ontology.Span, ontology.Section, ontology.Chapter}
* Owlready * Reparenting ontology.EU_32018L2001_Chapter_0_Section_0_Article_35_CUR9351_span: {ontology.Span} => {ontology.Article, ontology.Span, ontology.Section, ontology.Chapter}
* Owlready * Reparenting ontology.EU_32019L0944_Chapter_6_Section_3_A

In [10]:
# to reexamine, currently excluding:
# spans labeled with the feature value "end" and tag value ""
for inid, spid in no_tag_lst:
    for spanitm in pol_df.loc[inid, "Curation"]:
        if spanitm.span_id == spid:
            #print(f"\n{inid, spid}\n{spanitm.text}\n{spanitm.tag}\n{spanitm.feature}\n{spanitm.layer}")
            #print(spanitm)
            spanitm